# CineCue: 7. Explaining recommendations

Run notebooks 5 and 6 first. Each recommendation should come with a reason the user can
check: *"because you rated X."* That reason has to describe what the model actually
computed, not a plausible story added afterwards.

Our final ranking score is the blend from notebook 6:
(1 − α) · z(EASE) + α · z(MF). Both parts are linear in the user's history:

* **EASE:** the score is a sum of B[j, t] over the movies j the user rated. (The code
  also supports recency weights w_j; the final ranker from notebook 6 does not use them,
  so every w_j is 1 here.)
* **MF:** the user vector is a ridge solution A⁻¹Xᵀy. Its dot product with a movie
  vector splits into one term per rated movie (see `src/cinecue/explain.py`).

So every blended score is **exactly** a constant + a general-appeal term for the movie +
one term per rated movie. An explanation names the rated movies with the largest terms.
Explanations name movies the user rated **4 or 5 stars**, choosing those with the
largest positive terms. A movie the user disliked can also push a score up (EASE learns
from what people watch, whatever they thought of it), but "because you rated X 2★" would
confuse a reader. If no liked movie has a positive term, we name the largest positive term.
We then check two things:

1. **Exactness:** the terms rebuild the score to rounding error.
2. **Faithfulness:** removing the named movie from the history should hurt the
   recommendation more than removing a random rated movie. Removal also changes the
   per-user z-scaling and the MF ridge system, so this is a real counterfactual and not
   just the same arithmetic again.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Work from either the repository root or its notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "requirements.txt").is_file(), "Open this notebook from the cloned repository."
sys.path.insert(0, str(ROOT / "src"))
PROCESSED = ROOT / "data" / "processed"
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.4f}".format)

from cinecue.data import genre_columns, load_movies, load_splits
from cinecue.explain import blend_contributions
from cinecue.models import EASE, MatrixFactorization
from cinecue.ranking import blend, popularity_scale, recency_weights, rerank_by_popularity

splits = load_splits(PROCESSED)
history = pd.concat([splits["train"], splits["validation"]], ignore_index=True)
test = splits["test"]
catalog = load_movies(PROCESSED).set_index("movie_id")
GENRES = genre_columns(catalog)
ratings = json.loads((PROCESSED / "rating_summary.json").read_text(encoding="utf-8"))
ranking = json.loads((PROCESSED / "ranking_summary.json").read_text(encoding="utf-8"))
K, ALPHA, MIN_SUPPORT = ranking["k"], ranking["selected"]["alpha"], ranking["min_support"]
HALF_LIFE = ranking["selected"]["half_life_days"] or None  # 0 in the summary means no recency weighting
WEIGHT, EASE_REG = ranking["selected"]["rerank_weight"], ranking["selected"]["ease_reg"]
print(f"Blend alpha={ALPHA:g}, EASE reg={EASE_REG:g}, recency half-life={HALF_LIFE} days, re-rank weight={WEIGHT:g}")

mf_params = {key: ratings["mf"][key] for key in ["factors", "reg", "n_iter", "seed"]}
mf = MatrixFactorization(**mf_params).fit(history.user_id, history.movie_id, history.rating)
ease = EASE(reg=EASE_REG).fit(history.user_id, history.movie_id)
assert mf.movies.ids.equals(ease.movies.ids)
movie_ids = mf.movies.ids
counts = history.movie_id.value_counts().reindex(movie_ids).fillna(0).to_numpy()
popularity = popularity_scale(counts)
titles = catalog.loc[movie_ids, "title"].to_numpy()
genre_flags = catalog.loc[movie_ids, GENRES].to_numpy().astype(bool)
PRETTY = {"childrens": "Children's", "film_noir": "Film-Noir", "sci_fi": "Sci-Fi"}
genre_names = np.array([PRETTY.get(g[6:], g[6:].capitalize()) for g in GENRES])

Blend alpha=0.75, EASE reg=30000, recency half-life=None days, re-rank weight=1.5


## The recommender for one user

`recommend` rebuilds the final pipeline from notebook 6 for a single rating history:
recency-weighted EASE and MF scores, the per-user blend, the 20-rating support floor, then
popularity-aware re-ranking of the top 100.
It is the same computation that produced the test lists, applied one user at a time.

In [2]:
by_user = {user: group for user, group in history.groupby("user_id")}


NOW = history.timestamp.max()  # ranking time: the end of the validation period


def profile(user):
    """A user's rated movies, stars, and recency weights at the ranking time."""
    group = by_user[user]
    weights = recency_weights(group.timestamp.to_numpy(), NOW, HALF_LIFE)
    return mf.movies.encode(group.movie_id.to_numpy()), group.rating.to_numpy(dtype=float), weights


def recommend(positions, stars, weights, exclude=None):
    """Final top-K list, blended scores, and the excluded movies (rated or below the support floor)."""
    rated = ease.history_matrix(np.zeros(len(positions), int), positions, 1, weights)
    scores = blend(ease.score(rated), mf.score_items(mf.fold_in(positions, stars))[None, :], ALPHA)
    if exclude is None:
        # Exclude from the unweighted history, so tiny weights of old ratings never matter here.
        exclude = ease.history_matrix(np.zeros(len(positions), int), positions, 1).astype(bool)
        exclude |= (counts < MIN_SUPPORT)[None, :]
    return rerank_by_popularity(scores, exclude, popularity, WEIGHT, K)[0], scores[0], exclude


def named(row, stars, n=2):
    """Positions in the history of the movies an explanation names, strongest first."""
    order = np.argsort(-row, kind="stable")
    positive = order[row[order] > 0]
    liked = positive[stars[positive] >= 4]
    return (liked if len(liked) else positive)[:n]


liked = test.loc[test.rating >= ranking["relevant_stars"]]
eligible = np.array(sorted(set(liked.user_id) & set(by_user)))
rng = np.random.default_rng(7)
sample = np.sort(rng.choice(eligible, size=200, replace=False))
print(f"Explaining the top-{K} lists of {len(sample)} randomly chosen test users "
      f"(of {len(eligible):,} known users with a 4-5 star test rating).")

Explaining the top-10 lists of 200 randomly chosen test users (of 1,159 known users with a 4-5 star test rating).


## Exactness and concentration

In [3]:
records, cases = [], []
for user in sample:
    positions, stars, weights = profile(user)
    recs, scores, seen = recommend(positions, stars, weights)
    terms, appeal, constant = blend_contributions(ease, mf, ALPHA, positions, stars, recs, weights)
    rebuilt = constant + appeal + terms.sum(axis=1)
    for slot, movie in enumerate(recs):
        row = terms[slot]
        shown = named(row, stars)
        positive = row[row > 0].sum()
        records.append({"user_id": user, "slot": slot, "movie": movie, "error": abs(rebuilt[slot] - scores[movie]),
                        "history": len(row), "named_share": row[shown[0]] / positive,
                        "top3_share": np.sort(row)[::-1][:3].sum() / positive,
                        "named_stars": stars[shown[0]], "named_is_largest": shown[0] == np.argmax(row)})
    cases.append((user, positions, stars, weights, recs, scores, seen, terms, appeal))
records = pd.DataFrame(records)
max_error = records.error.max()
assert max_error < 1e-8, max_error
display(records[["history", "named_share", "top3_share", "named_stars"]].describe().loc[["mean", "50%", "min", "max"]])
display(Markdown(
    f"**Interpretation:** Over {len(records):,} recommendations, the terms rebuild every blended score to within "
    f"{max_error:.1e}, so the explanations are exact. The typical user has rated {records.history.median():.0f} movies, "
    f"and the push is spread across many of them: the named movie supplies a median "
    f"{records.named_share.median():.1%} of the positive personal push, and the three largest terms supply "
    f"{records.top3_share.median():.1%}. An explanation therefore names the strongest contributors, not a sole cause. "
    f"{(records.named_stars >= 4).mean():.1%} of named movies were rated 4-5 stars, and "
    f"{records.named_is_largest.mean():.1%} are also the single largest term overall."))

,history,named_share,top3_share,named_stars
mean,236.2300,0.0864,0.1941,4.9340
50%,156.0000,0.0401,0.1081,5.0000
min,5.0000,0.0045,0.0131,4.0000
max,2029.0000,0.8641,1.0000,5.0000


**Interpretation:** Over 2,000 recommendations, the terms rebuild every blended score to within 2.2e-15, so the explanations are exact. The typical user has rated 156 movies, and the push is spread across many of them: the named movie supplies a median 4.0% of the positive personal push, and the three largest terms supply 10.8%. An explanation therefore names the strongest contributors, not a sole cause. 100.0% of named movies were rated 4-5 stars, and 92.6% are also the single largest term overall.

## Faithfulness: remove the named movie

For every explained recommendation we refit the user's profile without one rated movie
and rebuild the whole list. Three conditions:

* **named movie:** remove the movie the explanation names first;
* **random:** remove a random other rated movie (a control);
* **most negative:** remove the movie with the most negative term. If the terms mean
  what they claim, the recommendation's score should *rise*.

The removed movie stays excluded from the list because the user has already rated it.

In [4]:
deletions = []
for user, positions, stars, weights, recs, scores, seen, terms, appeal in cases:
    for slot, movie in enumerate(recs):
        row = terms[slot]
        choices = {"named movie": int(named(row, stars)[0]), "most negative": int(np.argmin(row))}
        others = np.flatnonzero(np.arange(len(row)) != choices["named movie"])
        choices["random"] = int(rng.choice(others))
        for condition, drop in choices.items():
            keep = np.arange(len(positions)) != drop
            new_recs, new_scores, _ = recommend(positions[keep], stars[keep], weights[keep], seen)
            deletions.append({"condition": condition, "score_change": new_scores[movie] - scores[movie],
                              "left_top10": movie not in new_recs})
deletions = pd.DataFrame(deletions)
faithfulness = deletions.groupby("condition").agg(mean_score_change=("score_change", "mean"),
                                                  share_left_top10=("left_top10", "mean"),
                                                  cases=("score_change", "size"))
faithfulness = faithfulness.loc[["named movie", "random", "most negative"]]
display(faithfulness)
top, control = faithfulness.loc["named movie"], faithfulness.loc["random"]
display(Markdown(
    f"**Interpretation:** Removing the named movie lowers the recommendation's blended score by "
    f"{-top.mean_score_change:.3f} on average and pushes it out of the top 10 in {top.share_left_top10:.1%} of cases. "
    f"Removing a random rated movie changes the score by {control.mean_score_change:+.3f} and drops the movie in "
    f"{control.share_left_top10:.1%} of cases. Removing the most negative contributor changes the score by "
    f"{faithfulness.loc['most negative', 'mean_score_change']:+.3f}, in the predicted direction. The named movies "
    "really drive the recommendations. Because many rated movies push in the same direction, one removal often "
    "lowers a movie's rank without removing it from the list."))

,mean_score_change,share_left_top10,cases
condition,,,
named movie,-0.0874,0.1555,2000
random,0.0012,0.0315,2000
most negative,0.0353,0.0070,2000


**Interpretation:** Removing the named movie lowers the recommendation's blended score by 0.087 on average and pushes it out of the top 10 in 15.6% of cases. Removing a random rated movie changes the score by +0.001 and drops the movie in 3.1% of cases. Removing the most negative contributor changes the score by +0.035, in the predicted direction. The named movies really drive the recommendations. Because many rated movies push in the same direction, one removal often lowers a movie's rank without removing it from the list.

## What a user sees

Each line names up to two liked movies with the largest positive terms, the genres the
recommendation shares with the first one, MF's predicted rating, and whether re-ranking
moved the movie up. Every element comes from the computation above.

In [5]:
def explain(case, show=5):
    user, positions, stars, weights, recs, scores, seen, terms, appeal = case
    vector = mf.fold_in(positions, stars)
    candidates = np.flatnonzero(~seen[0])
    lines = [f"**User {user}** ({len(positions)} rated movies). Top {show} of {K}:"]
    for slot, movie in enumerate(recs[:show]):
        order = list(named(terms[slot], stars))
        anchors = " and ".join(f"*{titles[positions[j]]}* ({stars[j]:.0f}★)" for j in order)
        shared = genre_names[genre_flags[movie] & genre_flags[positions[order[0]]]] if order else []
        predicted = float(np.clip(mf.score_items(vector)[movie], 1, 5))
        plain_rank = int((scores[candidates] > scores[movie]).sum())
        note = f" Moved up from #{plain_rank + 1} because fewer people have rated it." if plain_rank > slot else ""
        genres = f" Shares {', '.join(shared)}." if len(shared) else ""
        lines.append(f"{slot + 1}. **{titles[movie]}**, because you rated {anchors}.{genres} "
                     f"Predicted rating {predicted:.1f}★.{note}")
    return "\n".join(lines)


for case in cases[:3]:
    display(Markdown(explain(case)))

**User 28** (91 rated movies). Top 5 of 10:
1. **American Beauty (1999)**, because you rated *Being John Malkovich (1999)* (4★) and *Shakespeare in Love (1998)* (5★). Shares Comedy. Predicted rating 4.0★.
2. **Godfather, The (1972)**, because you rated *Raiders of the Lost Ark (1981)* (5★) and *Alien (1979)* (5★). Shares Action. Predicted rating 4.2★.
3. **Schindler's List (1993)**, because you rated *Shakespeare in Love (1998)* (5★) and *Fargo (1996)* (4★). Predicted rating 4.2★.
4. **Casablanca (1942)**, because you rated *Graduate, The (1967)* (5★) and *Shakespeare in Love (1998)* (5★). Shares Drama, Romance. Predicted rating 4.2★.
5. **Blade Runner (1982)**, because you rated *Alien (1979)* (5★) and *Star Wars: Episode V - The Empire Strikes Back (1980)* (5★). Shares Sci-Fi. Predicted rating 4.1★.

**User 33** (196 rated movies). Top 5 of 10:
1. **Saving Private Ryan (1998)**, because you rated *Titanic (1997)* (5★) and *Shawshank Redemption, The (1994)* (5★). Shares Drama. Predicted rating 4.1★.
2. **Godfather: Part II, The (1974)**, because you rated *Godfather, The (1972)* (5★) and *GoodFellas (1990)* (5★). Shares Action, Crime, Drama. Predicted rating 4.1★. Moved up from #5 because fewer people have rated it.
3. **Terminator 2: Judgment Day (1991)**, because you rated *Jurassic Park (1993)* (5★) and *Star Wars: Episode VI - Return of the Jedi (1983)* (5★). Shares Action, Sci-Fi. Predicted rating 4.0★.
4. **Braveheart (1995)**, because you rated *Titanic (1997)* (5★) and *Jurassic Park (1993)* (5★). Shares Drama. Predicted rating 4.1★.
5. **E.T. the Extra-Terrestrial (1982)**, because you rated *Star Wars: Episode VI - Return of the Jedi (1983)* (5★) and *Titanic (1997)* (5★). Shares Sci-Fi. Predicted rating 4.0★.

**User 48** (518 rated movies). Top 5 of 10:
1. **Sanjuro (1962)**, because you rated *Breakfast at Tiffany's (1961)* (5★) and *X-Files: Fight the Future, The (1998)* (5★). Predicted rating 4.2★. Moved up from #12 because fewer people have rated it.
2. **For All Mankind (1989)**, because you rated *Star Wars: Episode I - The Phantom Menace (1999)* (5★) and *Star Wars: Episode VI - Return of the Jedi (1983)* (5★). Predicted rating 4.0★. Moved up from #37 because fewer people have rated it.
3. **Rear Window (1954)**, because you rated *Casablanca (1942)* (5★) and *Breakfast at Tiffany's (1961)* (5★). Predicted rating 4.1★.
4. **Before the Rain (Pred dozhdot) (1994)**, because you rated *U2: Rattle and Hum (1988)* (5★) and *Hard Day's Night, A (1964)* (5★). Predicted rating 3.9★. Moved up from #85 because fewer people have rated it.
5. **Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963)**, because you rated *Casablanca (1942)* (5★) and *Hard Day's Night, A (1964)* (5★). Shares War. Predicted rating 3.9★.

## Users with no history

A user who is new at the cutoff has no ratings to point to, so the honest explanation
is about the movie itself: how many people rated it and how well it was received. As
soon as the user rates movies, the profile updates (notebook 5) and personal
explanations become available.

In [6]:
new_list = np.argsort(-counts, kind="stable")[:5]
lines = ["**A new user** sees the most-rated movies:"]
for slot, movie in enumerate(new_list):
    mean_stars = history.loc[history.movie_id == movie_ids[movie], "rating"].mean()
    lines.append(f"{slot + 1}. **{titles[movie]}**: popular with other viewers ({counts[movie]:,.0f} ratings, "
                 f"average {mean_stars:.1f}★).")
display(Markdown("\n".join(lines)))

**A new user** sees the most-rated movies:
1. **American Beauty (1999)**: popular with other viewers (3,275 ratings, average 4.3★).
2. **Star Wars: Episode V - The Empire Strikes Back (1980)**: popular with other viewers (2,832 ratings, average 4.3★).
3. **Star Wars: Episode IV - A New Hope (1977)**: popular with other viewers (2,823 ratings, average 4.5★).
4. **Star Wars: Episode VI - Return of the Jedi (1983)**: popular with other viewers (2,756 ratings, average 4.0★).
5. **Jurassic Park (1993)**: popular with other viewers (2,542 ratings, average 3.8★).

In [7]:
summary = {"users": int(len(sample)), "recommendations": int(len(records)),
           "max_reconstruction_error": float(max_error),
           "median_history": float(records.history.median()),
           "median_named_share": float(records.named_share.median()),
           "median_top3_share": float(records.top3_share.median()),
           "named_liked_share": float((records.named_stars >= 4).mean()),
           "named_is_largest_share": float(records.named_is_largest.mean()),
           "deletion": faithfulness.reset_index().to_dict(orient="records")}
(PROCESSED / "explanation_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print("Saved explanation_summary.json to data/processed/.")

Saved explanation_summary.json to data/processed/.


**Limits.** The explanations describe the model, not the user's real motives. A named
movie is the one that moved the score most, which is not necessarily why the person
would enjoy the recommendation. MovieLens has no feedback on explanations, so we cannot
measure whether users find them persuasive or trustworthy. That would need a user study.
Phase 2 ends here.